# 03b - AfriMT5 Fine-Tuning with Official Validation Set
## Multilingual Health QA - MLT1 Final Project | Phase 2

### Overview
This notebook improves on the previous fine-tuning pipeline (`03_finetuning_experiments`) in two key ways:

1. **Better base model:** `masakhane/afri-mt5-base` - mT5 continued-pretrained on 17 African languages, providing stronger representations for Akan, Luganda, Kiswahili, and Amharic compared to vanilla `google/mt5-base`.

2. **Official validation set:** Instead of a self-made 90/10 train split, we use the full `Train.csv` (29,814 examples) for training and the competition-provided `Val.csv` (6,686 examples) for validation - giving us ~3,000 more training examples and a more reliable evaluation signal.

### Key findings from Phase 1 (03_finetuning_experiments)
- Zero-shot mT5-small: ROUGE-1 = 0.0025, Zindi = 0.000676
- Fine-tuned mT5-small + LoRA (Exp 2/3): ROUGE-1 = 0.1520, Zindi = 0.145043
- Fine-tuned mT5-base + LoRA (Exp 7): ROUGE-1 = 0.1928, Zindi = 0.211963
- AfriMT5-base + LoRA (fp16, collapsed): ROUGE-1 = 0.2337 (val only, not submitted)
- AfriMT5-base + LoRA (fp32): ROUGE-1 = 0.2395, Zindi = 0.160617 (fp32 hurt inference)
- AfriMT5-base + LoRA (AdamW, fp16): Training in progress

### Strategy for this notebook
- Model: `masakhane/afri-mt5-base`
- LoRA: r=32, alpha=64, target_modules=["q","v"]
- Optimizer: AdamW (stable under fp16, unlike Adafactor)
- Precision: fp16=True with max_grad_norm=1.0
- Epochs: 3 (balance between underfitting at 1 epoch and overfitting at 5)
- Training data: Full Train.csv (29,814 examples)
- Validation data: Official Val.csv (6,686 examples)
- Evaluation: Local ROUGE on Val.csv + Zindi leaderboard submissions

In [5]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, MT5ForConditionalGeneration
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from rouge_score import rouge_scorer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DATA_DIR = '/kaggle/input/datasets/jokjohnkur/health-qa-data-v5'
train = pd.read_csv(f'{DATA_DIR}/Train.csv')
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

train = train[train['input'].str.strip() != ''].reset_index(drop=True)
val   = val[val['input'].str.strip() != ''].reset_index(drop=True)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

Device: cuda
Train: 29814 | Val: 6686 | Test: 2618


### Load AfriMT5 + LoRA

In [6]:
tokenizer = AutoTokenizer.from_pretrained("masakhane/afri-mt5-base")
model = MT5ForConditionalGeneration.from_pretrained("masakhane/afri-mt5-base").to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q", "v"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("AfriMT5 + LoRA ready")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 3,538,944 || all params: 970,112,256 || trainable%: 0.3648
AfriMT5 + LoRA ready


### Building datasets with proper label masking + language-aware prompts

In [7]:
subset_to_language_name = {
    'Aka_Gha': 'Akan', 'Amh_Eth': 'Amharic',
    'Eng_Eth': 'English', 'Eng_Gha': 'English',
    'Eng_Ken': 'English', 'Eng_Uga': 'English',
    'Lug_Uga': 'Luganda', 'Swa_Ken': 'Kiswahili',
}

def build_prompt(question, language=None):
    if language and language in subset_to_language_name:
        lang_name = subset_to_language_name[language]
        return f'Answer this health question in {lang_name}: {question}'
    return f'Answer this health question: {question}'

def make_hf_dataset(df):
    records = []
    for _, row in df.iterrows():
        prompt = build_prompt(str(row['input']), str(row['subset']))
        records.append({'prompt': prompt, 'answer': str(row['output'])})

    raw_ds = Dataset.from_list(records)

    def preprocess(examples):
        model_inputs = tokenizer(
            examples['prompt'],
            max_length=256, truncation=True, padding=False,
        )
        labels = tokenizer(
            text_target=examples['answer'],
            max_length=512, truncation=True, padding=False,
        )
        # Mask padding tokens so loss ignores them
        model_inputs['labels'] = [
            [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
            for seq in labels['input_ids']
        ]
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])

print("Building datasets...")
hf_train = make_hf_dataset(train)
hf_val   = make_hf_dataset(val)
print(f"Train: {len(hf_train)} | Val: {len(hf_val)}")

Building datasets...


Map:   0%|          | 0/29814 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train: 29814 | Val: 6686


### Training

In [13]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Auto-detect bf16 vs fp16
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"bf16: {use_bf16} | fp16: {use_fp16}")

training_args = Seq2SeqTrainingArguments(
    output_dir="./afrimt5-final",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    bf16=use_bf16,
    fp16=use_fp16,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    generation_max_length=512,
    max_grad_norm=1.0,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
)

trainer.train()

bf16: True | fp16: False


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,22.613936,4.783440


TrainOutput(global_step=1864, training_loss=24.090126266806934, metrics={'train_runtime': 9870.391, 'train_samples_per_second': 3.021, 'train_steps_per_second': 0.189, 'total_flos': 4478019087974400.0, 'train_loss': 24.090126266806934, 'epoch': 1.0})

In [14]:
model.eval()

for i in range(3):
    row = val.iloc[i]
    lang = subset_to_language_name.get(str(row['subset']), 'English')
    prompt = f'Answer this health question in {lang}: {row["input"]}'
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"[{row['subset']}] Q: {row['input'][:80]}")
    print(f"Pred: {pred[:150]}")
    print()

[Aka_Gha] Q: Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronk
Pred: Nkɔmmɔbɔ ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn a wote beae a ntawantawa wɔ mu denam: Ns

[Aka_Gha] Q: Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?
Pred: Ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase, a ɛfa nsɛm a wɔde bɛhyɛ nkɔmmɔbɔ ne akwahosan ho nhyehyɛe ahorow a wobɛhwehwɛ so.

[Aka_Gha] Q: Mɛyɛ dɛn atumi abɔ asisifo ho amanneɛ wɔ ɔkwan a etu mpɔn na ahobammɔ wom so, na
Pred: Ɔkwan a etu mpɔn na ahobammɔ wom so, anammɔn bɔ sɛ wɔfa m’amanneɛbɔ no aniberesɛm denam nsɛm a ɛwɔ ɔkwan ho amanneɛ a wɔde bɛhwehwɛ akwahosan ho dwuma



In [15]:
import re
import time

model.eval()
test_predictions_afrimt5 = []
start = time.time()

for idx, row in test.iterrows():
    lang = subset_to_language_name.get(str(row['subset']), 'English')
    prompt = f'Answer this health question in {lang}: {row["input"]}'
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = re.sub(r'<extra_id_\d+>', '', pred).strip()
    test_predictions_afrimt5.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)} — {(time.time()-start)/60:.1f} min")

print(f"Done! {(time.time()-start)/60:.1f} min total")

Processed 0/2618 — 0.1 min
Processed 500/2618 — 46.2 min
Processed 1000/2618 — 69.7 min
Processed 1500/2618 — 85.0 min
Processed 2000/2618 — 98.4 min
Processed 2500/2618 — 124.3 min
Done! 129.3 min total


In [16]:
import pandas as pd
import re

submission_afrimt5 = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': [re.sub(r'<extra_id_\d+>', '', p).strip() for p in test_predictions_afrimt5],
    'TargetR1F1': [re.sub(r'<extra_id_\d+>', '', p).strip() for p in test_predictions_afrimt5],
    'TargetLLM':  [re.sub(r'<extra_id_\d+>', '', p).strip() for p in test_predictions_afrimt5],
})

submission_afrimt5.to_csv('submission_afrimt5_v2.csv', index=False)
print("Saved!")
submission_afrimt5.head(3)

Saved!


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nneɛma a wode bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...","Nneɛma a wode bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...","Nneɛma a wode bɛyɛ nkyerɛkyerɛ nneɛma, adwumay..."
1,ID_TS_Aka_Gha_1C80317F,Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...,Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...,Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...
2,ID_TS_Aka_Gha_06671AD1,Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...,Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...,Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...
